<a href="https://colab.research.google.com/github/alex-tianhuang/idrfeatlib/blob/main/notebooks/FeatureDistances.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Setup environment by downloading and installing the idrfeatlib repo.
#
# You only need to run this once.
!file idrfeatlib/ >/dev/null && rm -rf idrfeatlib
!git clone https://github.com/alex-tianhuang/idrfeatlib --quiet
%pip install idrfeatlib/

Processing ./idrfeatlib
  Preparing metadata (setup.py) ... done
  Created wheel for idrfeatlib: filename=idrfeatlib-0.0.0-py3-none-any.whl size=31311 sha256=0871e1713f358e2ba6c85d875f63b66c4f00d4d0ead59dfd3bd13dc0722f0a40
  Stored in directory: /tmp/pip-ephem-wheel-cache-uxztdolw/wheels/d5/cf/10/5dfdc8ed2b6a9afbf9abd3de203724f1fd35094f1eb11f5312
Successfully built idrfeatlib


In [10]:

def main(args):
    """
    Compute the feature distance between each sequence
    in a list of sequences and a reference sequence.

    Input sequences in FASTA format

    Output CSV format is like:
    ```
    ProteinID,Distance
    Protein-1,0.5
    Protein-2,0.7
    ```
    """
    import csv
    import json
    import sys
    from idrfeatlib.utils import read_fasta
    from idrfeatlib import FeatureVector
    from idrfeatlib.metric import Metric
    from idrfeatlib.featurizer import compile_featurizer, Featurizer
    from idrfeatlib.native import compile_native_featurizer
    for label, feature_vector in FeatureVector.load(args.feature_weights_file):
        if label == args.weights_feature_vector:
            metric = Metric(feature_vector, feature_vector)
            break
    else:
        raise RuntimeError("could not find feature vector `%s` in %s" % (args.weights_feature_vector, args.feature_weights_file))

    featurizer, errors = compile_native_featurizer()
    for featname, error in errors.items():
        print("error compiling `%s`: %s" % (featname, error), file=sys.stderr)
    if featurizer.keys() != metric.weights.as_dict.keys():
        raise RuntimeError("featurizer and metric feature vector `%s` have different features" % args.weights_feature_vector)

    rows = read_fasta(args.input_sequences)
    ref_sequence = args.ref_seq
    featurizer = Featurizer(featurizer)
    try:
        ref_fvec, _ = featurizer.featurize(ref_sequence, acceptable_errors=())
    except (ArithmeticError, ValueError, KeyError) as e:
        print("features of reference sequence could not be computed", file=sys.stderr)
        print(e, file=sys.stderr)
        return
    fvecs_all, errors_all = featurizer.featurize_to_matrices(rows)
    output = []
    for protein_id, _ in rows:
        if (errors := errors_all.get(protein_id)) is not None:
            print(f"failed to compute feature distance for {protein_id}:", file=sys.stderr)
            for feature_id, error in errors.items():
                print(f"  {feature_id}: {error}", file=sys.stderr)
            continue
        output.append({
            "ProteinID": protein_id,
            "Distance": metric.euclidean_distance_between(fvecs_all[protein_id], ref_fvec),
        })
    with open(args.output_file, "w") as file:
        w = csv.DictWriter(file, ["ProteinID", "Distance"])
        w.writeheader()
        w.writerows(output)


def display_csv(output_name):
    """
    Show the table in the notebook.

    I assume colab will forever keep pandas as available by default.
    """
    from IPython.display import display
    import pandas as pd

    df = pd.read_csv(output_name)

    print()
    print("Showing output below")
    print("--------------------")
    print()
    display(df)
    print()

def run_colab_wrapper(output_name: str):
    """Design a sequence to match an arbitrary feature vector given in a csv."""
    import argparse
    import os
    from google.colab import files
    from idrfeatlib.featurizer import Featurizer

    args = argparse.Namespace()

    args.ref_seq = input("Please paste in your reference sequence: ")

    args.input_sequences = 'input_sequences.fasta'
    goto_upload = True
    if os.path.exists(args.input_sequences):
        choice = input(f"The file {args.input_sequences} already exists. Would you like to overwrite it? (y/n)")
        if choice.lower() != 'y':
            goto_upload = False
    if goto_upload:
        files.upload_file(args.input_sequences)

    print("How would you like to weight the features?")
    print("1. Use inv-std from all human IDRs.")
    print("2. Use inv-std from Disprot.")
    print("3. Provide a custom CSV of feature weights (provide file).")
    choice = input("Enter choice (1, 2, or 3): ")
    if choice not in ['1', '2', '3']:
        print("Invalid choice. Defaulting to inv-std from all human IDRs.")
        choice = '1'
    args.weights_feature_vector = "weights"
    if choice == '1':
        args.feature_weights_file = 'idrfeatlib/notebooks/data/inv_std_human.csv'
    elif choice == '2':
        args.feature_weights_file = 'idrfeatlib/notebooks/data/inv_std_disprot.csv'
    elif choice == '3':
        args.feature_weights_file = 'feature_weights.csv'
        print("Please upload a CSV of feature weights.")
        files.upload_file(args.feature_weights_file)

    args.output_file = output_name
    if os.path.exists(args.output_file):
        overwrite = input(f"Output file '{args.output_file}' already exists. Overwrite? (y/n)").lower()
        if overwrite != 'y':
            raise RuntimeError(f"Output file '{args.output_file}' already exists. Please rename or delete it.")
        print(f"Output file '{args.output_file}' overwritten.")

    main(args)
    display_csv(args.output_file)
    print(f"Downloading output file to {args.output_file}")
    files.download(args.output_file)

In [11]:
run_colab_wrapper("feature_distances.csv")
print("Done!")

Please paste in your reference sequence: MLFRNIEVGRQAAKLLTRTSSRLAWQSIGASRNISTIRQQIRKTQLYNFKKTVSIRPFSLSSPVFKPHVASESNPIESRLKTSKNVAYWLIGTSGLVFGIVVLGGLTRLTESGLSITEWKPVTGTLPPMNQKEWEEEFIKYKESPEFKLLNSHIDLDEFKFIFFMEWIHRLWGRAIGAVFILPAVYFAVSKKTSGHVNKRLFGLAGLLGLQGFVGWWMVKSGLDQEQLDARKSKPTVSQYRLTTHLGTAFFLYMGMLWTGLEILRECKWIKNPVQAISLFKKLDNPAIGPMRKISLALLAVSFLTAMSGGMVAGLDAGWVYNTWPKMGERWFPSSRELMDENFCRREDKKDLWWRNLLENPVTVQLVHRTCAYVAFTSVLAAHMYAIKKKAVIPRNAMTSLHVMMGVVTLQATLGILTILYLVPISLASIHQAGALALLTSSLVFASQLRKPRAPMRNVIITLPHSSKVTSGKILSEASKLASKPL
The file input_sequences.fasta already exists. Would you like to overwrite it? (y/n)n
How would you like to weight the features?
1. Use inv-std from all human IDRs.
2. Use inv-std from Disprot.
3. Provide a custom CSV of feature weights (provide file).
Enter choice (1, 2, or 3): 1
Output file 'feature_distances.csv' already exists. Overwrite? (y/n)y
Output file 'feature_distances.csv' overwritten.


failed to compute feature distance for 54:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 55:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 56:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 57:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 58:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 59:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 60:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 61:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 62:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 63:
  arospacing: sequence has no aromatic residues
failed to compute feature distance for 64:
  arospacing: sequence has no aromatic residues


Showing output below
--------------------



,ProteinID,Distance
0,1,334.969591
1,2,346.526436
2,3,402.090217
3,4,272.524141
4,5,371.432350
...,...,...
63,80,614.317564
64,81,405.287389
65,82,384.035160
66,83,741.184682


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
